In [0]:
%python
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from pyspark.sql.functions import current_timestamp

# 1. Configuramos MLflow para registrar modelos directamente en Unity Catalog
mlflow.set_registry_uri("databricks-uc")
model_name = "logistics_prod.models.delay_risk_predictor"

# 2. Creamos el Feature Dataset cruzando Silver IoT + Silver SQL Server ERP
query = """
SELECT
    t.event_timestamp,
    t.asset_id,
    f.cargo_category,
    f.assigned_hub,
    t.temperature_c,
    f.max_allowed_temp_c,
    (t.temperature_c - f.max_allowed_temp_c) AS temp_margin_c,
    t.humidity_pct,
    t.waiting_time_mins,
    t.asset_utilization_pct,
    CASE
        WHEN t.traffic_status = 'Heavy' THEN 2
        WHEN t.traffic_status = 'Moderate' THEN 1
        ELSE 0
    END AS traffic_encoded,
    t.is_delayed
FROM logistics_prod.silver.logistics_clean t
INNER JOIN logistics_prod.silver.erp_fleet_assets_clean f
    ON t.asset_id = f.asset_id
"""

df_spark = spark.sql(query)
pdf = df_spark.toPandas()

# 3. Preparamos variables predictoras (Features) y variable objetivo (Target)
feature_cols = [
    "temperature_c", "max_allowed_temp_c", "temp_margin_c",
    "humidity_pct", "waiting_time_mins", "asset_utilization_pct", "traffic_encoded"
]
X = pdf[feature_cols].fillna(0)
y = pdf["is_delayed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Entrenamos y registramos el experimento en MLflow + Unity Catalog
with mlflow.start_run(run_name="rf_delay_predictor_v1") as run:
    clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    # Guardamos métricas y parámetros en MLflow
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("test_accuracy", acc)
    mlflow.log_metric("test_roc_auc", auc)

    # Inferimos la firma de entrada/salida (obligatorio para Unity Catalog)
    signature = infer_signature(X_train, preds)

    # Registramos el modelo en Unity Catalog
    mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="random_forest_model",
        signature=signature,
        registered_model_name=model_name
    )
    print(f"Modelo registrado en Unity Catalog: {model_name} | Accuracy: {acc:.2%} | ROC-AUC: {auc:.3f}")

# 5. Inferencia Batch: Generamos probabilidades de riesgo para toda la flota en Gold
pdf["predicted_delay"] = clf.predict(X)
pdf["delay_probability_pct"] = (clf.predict_proba(X)[:, 1] * 100).round(2)

pred_spark_df = (
    spark.createDataFrame(pdf[[
        "event_timestamp", "asset_id", "cargo_category", "assigned_hub",
        "temperature_c", "max_allowed_temp_c", "waiting_time_mins",
        "is_delayed", "predicted_delay", "delay_probability_pct"
    ]])
    .withColumn("scored_at", current_timestamp())
)

pred_spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "logistics_prod.gold.ml_delay_predictions"
)
print("Tabla Gold de predicciones ML 'logistics_prod.gold.ml_delay_predictions' creada con éxito.")
